# OscilloSim Quickstart

This notebook demonstrates PRINet's `OscilloSim` interactive oscillator
simulator. We'll explore Kuramoto synchronization, coupling topologies,
and chimera states.

**Prerequisites:** Install PRINet from source (`pip install -e ".[dev]"`)


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from prinet.utils.oscillosim import (
    OscilloSim,
    quick_simulate,
    SimulationResult,
    ring_topology,
    small_world_topology,
    local_order_parameter,
    chimera_index,
)
from prinet import kuramoto_order_parameter

print(f"PRINet imported successfully, torch {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Quick Simulation

The fastest way to run a Kuramoto simulation:

In [ ]:
result = quick_simulate(
    n_oscillators=10_000,
    n_steps=200,
    coupling_strength=2.0,
    device=DEVICE,
    seed=42,
)

print(f"Oscillators: {result.n_oscillators:,}")
print(f"Steps:       {result.n_steps}")
print(f"Coupling:    {result.coupling_mode}")
print(f"Final r:     {result.order_parameter[-1]:.4f}")
print(f"Wall time:   {result.wall_time_s:.3f} s")
print(f"Throughput:  {result.throughput:,.0f} osc-steps/s")

## 2. Synchronization Dynamics

Record the full trajectory to visualize how the order parameter evolves:

In [ ]:
sim = OscilloSim(
    n_oscillators=1000,
    coupling_strength=2.0,
    coupling_mode="mean_field",
    device=DEVICE,
    seed=42,
)

result = sim.run(n_steps=500, dt=0.01, record_trajectory=True, record_interval=5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Order parameter over time
axes[0].plot(result.order_parameter)
axes[0].set_xlabel("Recording step")
axes[0].set_ylabel("r (order parameter)")
axes[0].set_title("Synchronization")
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# Final phase distribution
final_phase = result.final_phase.cpu().numpy()
axes[1].hist(final_phase, bins=50, density=True, alpha=0.7)
axes[1].set_xlabel("Phase (rad)")
axes[1].set_ylabel("Density")
axes[1].set_title(f"Final phase distribution (r={result.order_parameter[-1]:.3f})")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Coupling Strength Sweep

The Kuramoto model has a critical coupling strength $K_c$ above which
synchronization emerges. Let's sweep $K$ to find it:

In [ ]:
K_values = np.linspace(0.0, 5.0, 25)
final_r = []

for K in K_values:
    sim = OscilloSim(
        n_oscillators=2000,
        coupling_strength=float(K),
        coupling_mode="mean_field",
        device=DEVICE,
        seed=42,
    )
    res = sim.run(n_steps=300, dt=0.01)
    final_r.append(res.order_parameter[-1])

plt.figure(figsize=(7, 4))
plt.plot(K_values, final_r, "o-", markersize=4)
plt.xlabel("Coupling strength K")
plt.ylabel("Final order parameter r")
plt.title("Kuramoto Phase Transition")
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="r=0.5")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Coupling Topologies

Compare mean-field, ring, sparse k-NN, and small-world coupling:

In [ ]:
N = 1000
modes = ["mean_field", "ring", "sparse_knn", "small_world"]

fig, ax = plt.subplots(figsize=(8, 4))

for mode in modes:
    sim = OscilloSim(
        n_oscillators=N,
        coupling_strength=3.0,
        coupling_mode=mode,
        k_neighbors=10,
        p_rewire=0.1,
        device=DEVICE,
        seed=42,
    )
    res = sim.run(n_steps=500, dt=0.01, record_trajectory=True, record_interval=5)
    ax.plot(res.order_parameter, label=mode)

ax.set_xlabel("Recording step")
ax.set_ylabel("r")
ax.set_title("Synchronization by Coupling Topology")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Chimera States

Chimera states occur when part of the network synchronizes while the
rest remains incoherent. We use a ring topology with moderate coupling:

In [ ]:
N_chimera = 512
sim = OscilloSim(
    n_oscillators=N_chimera,
    coupling_strength=1.5,
    coupling_mode="ring",
    k_neighbors=6,
    device=DEVICE,
    seed=7,
)

res = sim.run(n_steps=2000, dt=0.005)

# Compute local order parameter
nbr = ring_topology(N_chimera, 6, device=DEVICE)
local_r = local_order_parameter(res.final_phase, nbr).cpu().numpy()
chi = chimera_index(res.final_phase, nbr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(range(N_chimera), res.final_phase.cpu().numpy(),
                s=2, c=local_r, cmap="coolwarm", vmin=0, vmax=1)
axes[0].set_xlabel("Oscillator index")
axes[0].set_ylabel("Phase (rad)")
axes[0].set_title(f"Phase snapshot (chimera index = {chi:.3f})")
axes[0].colorbar = plt.colorbar(
    axes[0].collections[0], ax=axes[0], label="Local r")

axes[1].plot(local_r)
axes[1].axhline(0.5, color="red", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Oscillator index")
axes[1].set_ylabel("Local order parameter")
axes[1].set_title("Coherence profile")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

| Feature | Demonstrated |
|---------|-------------|
| `quick_simulate` | One-line simulation |
| Trajectory recording | `record_trajectory=True` |
| Phase transition | $K$ sweep |
| Topology comparison | mean_field, ring, sparse_knn, small_world |
| Chimera detection | `local_order_parameter`, `chimera_index` |

See the [API docs](https://prinet.readthedocs.io) for the full
`OscilloSim` reference.